# ⚡ GridMap Web Server on Google Colab (12 GB Free Cloud RAM)

This notebook runs the **GridMap Utility Web Server** inside Google Cloud's 12.7 GB RAM environment with an instant public HTTPS tunnel. Keep this tab open in the background so the tunnel stays live!

In [ ]:
# 1. Clone or update repository
!git clone https://github.com/anushreya0606/gridmap-utility.git /content/gridmap-utility 2>/dev/null || (cd /content/gridmap-utility && git pull)
%cd /content/gridmap-utility

# 2. Install requirements
!pip install -q -r requirements.txt waitress

# 3. Install Cloudflare tunnel for Linux
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 4. Launch web server in background with isolated public workspace mode
import os, subprocess, time, re

env = os.environ.copy()
env["GRIDMAP_PUBLIC"] = "1"
env["GRIDMAP_KEEP_DAYS"] = "7"

server_proc = subprocess.Popen(["python", "app.py", "--host", "127.0.0.1", "--port", "5000"], env=env)
time.sleep(3)

# 5. Start public tunnel
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("Connecting to Google Cloud and creating public tunnel...")
for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print("\n" + "="*75)
        print("🎉 YOUR WEBSITE IS LIVE ON GOOGLE CLOUD (12 GB RAM)!")
        print(f"👉 PUBLIC LINK: {url}")
        print("="*75 + "\n")
        print("🟢 Server is actively running. Keep this Colab tab running.")
        break

# Keep the process active so Colab does not terminate the tunnel
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("Stopping...")
    server_proc.terminate()
    tunnel_proc.terminate()
